# **Dự Đoán Chất Lượng Không Khí Cho Ngày Hôm Sau Ở TP.HCM**

Chất lượng không khí tại TP. Hồ Chí Minh ngày càng trở thành mối quan tâm lớn của người dân, đặc biệt với chỉ số AQI trung bình năm 2022-2026 đạt 81.6 (mức Trung bình theo thang US), trong đó nhiều ngày vượt ngưỡng 100 - mức Không tốt cho nhóm nhạy cảm. Khả năng dự báo AQI trước một ngày giúp người dân lên kế hoạch sinh hoạt, hạn chế phơi nhiễm, và hỗ trợ các cơ quan quản lý môi trường.

### **Mục tiêu notebook này:**

- Xây dựng và huấn luyện các mô hình Machine Learning để dự đoán mức độ chất lượng không khí ngày hôm sau (bài toán Classification) - bổ sung cho bài toán Regression.
- So sánh hiệu suất giữa các mô hình phân loại: Random Forest Classifier, XGBoost Classifier và LightGBM Classifier.
- Phát hiện và xử lý vấn đề mất cân bằng dữ liệu - phần lớn các ngày rơi vào mức "Moderate", trong khi các mức "Good" và "Unhealthy" chiếm tỷ lệ rất nhỏ
- Đánh giá mô hình bằng các chỉ số phù hợp với bài toán phân loại đa lớp: Accuracy, Precision, Recall, F1-score, Confusion Matrix.
- Xây dựng bảng khuyến nghị hành động cụ thể tương ứng với từng mức AQI dự đoán
- Lựa chọn Best Classifier và lưu lại (`best_classifier.pkl`) để sử dụng trong `predict.py` và Dashboard.
- Tạo cơ chế "2 mô hình đồng thuận": Kết hợp kết quả Classification với kết quả Regression để tăng độ tin cậy khi dự đoán thực tế.

## **00. Import và cấu hình**

In [ ]:
# Standard Libraries
import warnings
import gdown
import os
import pickle
import joblib
import json
import copy

from google.colab import drive
from datetime import datetime

# Data Manipulation - Math
import numpy as np
import pandas as pd

# Data Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import folium
from matplotlib.colors import LinearSegmentedColormap
from statsmodels.tsa.seasonal import seasonal_decompose
from scipy.stats import gaussian_kde
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.colorbar import ColorbarBase

# Machine Learning - Preprocessing
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
# Cấu hình
warnings.filterwarnings(
    "ignore"
)  # Tắt các cảnh báo không cần thiết cho notebook sạch hơn
pd.set_option(
    "display.float_format", "{:.2f}".format
)  # Hiển thị số thập phân gọn hơn (làm tròn 2 chữ số thập phân)
plt.rcParams.update(
    {
        "figure.dpi": 150,
        "axes.grid": True,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.family": "DejaVu Sans",
    }
)  # Cấu hình style mặc định cho toàn bộ biểu đồ matplotlib trong notebook

# AQI Color Palette
# Đây là màu chính thức của US EPA, để đảm bảo nhất quán toàn bộ notebook
AQI_COLORS = {
    "Good": "#00E400",
    "Moderate": "#FFFF00",
    "Unhealthy for Sensitive Groups": "#FF7E00",
    "Unhealthy": "#FF0000",
    "Very Unhealthy": "#8F3F97",
    "Hazardous": "#7E0023",
}

AQI_BINS = [0, 50, 100, 150, 200, 300, 500]
AQI_LABELS = list(AQI_COLORS.keys())
AQI_MAPPING = {label: idx for idx, label in enumerate(AQI_LABELS)}


# Hàm AQI_CATEGORY()
def AQI_CATEGORY(value):
    """
    Phân loại mức AQI theo thang US EPA
    Input : Giá trị AQI (số thực)
    Output: Tên mức (string)
    """
    if value <= 50:
        return "Good"
    elif value <= 100:
        return "Moderate"
    elif value <= 150:
        return "Unhealthy for Sensitive Groups"
    elif value <= 200:
        return "Unhealthy"
    elif value <= 300:
        return "Very Unhealthy"
    else:
        return "Hazardous"


# Tạo Custom Gradient Colormap từ AQI_COLORS và AQI_BINS
# Chuẩn hóa các mốc AQI vì LinearSegmentedColormap là thang đo 0.0 đến 1.0
MAX_AQI = 500
positions = [
    val / MAX_AQI for val in AQI_BINS
]  # positions sẽ trở thành [0.0, 0.1, 0.2, 0.3, 0.4, 0.6, 1.0]

# Vì AQI_BINS có 7 mốc nhưng AQI_COLORS chỉ có 6 màu
colors = list(AQI_COLORS.values())
gradient_colors = colors + [colors[-1]]  # Nhân đôi màu cuối cùng

# Ghép vị trí và màu sắc lại để tạo dải gradient
color_mapping = list(
    zip(positions, gradient_colors)
)  # Gán các màu tương ứng với từng giá trị
AQI_gradient_cmap = LinearSegmentedColormap.from_list(
    "AQI_gradient", color_mapping
)  # Tạo ra dãy màu liên tục thay vì riêng lẻ

In [ ]:
# drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/HCMUS/Nhập Môn Khoa Học Dữ Liệu/Mini Project"

In [ ]:
folder_ID = "1b8LGMtOwMrj6FDMODA8-rJNq0cKXlgBA"
folder_url = f"https://drive.google.com/drive/folders/{folder_ID}"

gdown.download_folder(folder_url, output="data", quiet=False)

X_train = pd.read_csv("data/X_train.csv", index_col="date", parse_dates=True)
X_test = pd.read_csv("data/X_test.csv", index_col="date", parse_dates=True)
y_train = pd.read_csv("data/y_train.csv", index_col="date", parse_dates=True)
y_test = pd.read_csv("data/y_test.csv", index_col="date", parse_dates=True)

y_train_cls = y_train["target_class_tomorrow"]
y_test_cls = y_test["target_class_tomorrow"]

## **04. Model Classification**



### **4.1. Phân tích mất cân bằng**

In [ ]:
# Kiểm tra class balance
df = pd.concat([X_train, X_test], axis=0)
df["target"] = pd.concat([y_train_cls, y_test_cls], axis=0)

df["target"].value_counts(normalize=True).sort_index().plot(
    kind="bar", xlabel="Pollution Level", ylabel="Frequency", title="Class Balance"
)
plt.xticks(rotation=0)
plt.grid(False)
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "imbalanced_data")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

#### **Phân tích hiện trạng mất cân bằng dữ liệu (Imbalanced Data)**

- **Thực trạng từ biểu đồ Class Balance:** Tập dữ liệu đang đối mặt với tình trạng mất cân bằng phân lớp cực kỳ nghiêm trọng, bao gồm 4 nhãn (0, 1, 2, 3). Trong đó:

Nhãn **1** chiếm tỷ lệ áp đảo tuyệt đối lên tới xấp xỉ **76.5%**.

Nhãn **2** chiếm khoảng **18%**.

Các nhãn **0** và **3** là những trường hợp đặc biệt, lần lượt chỉ chiếm khoảng **4%** và **1%**.

- **Lý do thực tế tại TP.HCM:**
Đặc thù khí hậu ổn định kết hợp với lượng khí thải giao thông/công nghiệp duy trì đều đặn khiến phần lớn các ngày trong năm neo ở một ngưỡng chất lượng không khí nhất định (Nhãn 1). Những ngày có không khí cực kỳ trong lành (sau mưa bão) hoặc ô nhiễm nghiêm trọng hiếm khi xảy ra.

- **Lựa chọn chiến lược xử lý:**
Với phân bố có lớp thiểu số chỉ chiếm 1%, việc sử dụng các kỹ thuật tái lấy mẫu (Resampling) như Undersampling sẽ làm mất mát gần như toàn bộ dữ liệu, trong khi SMOTE (Oversampling) dễ sinh ra nhiễu do thiếu điểm gốc.

> Do đó, giải pháp tối ưu và an toàn nhất là sử dụng **Trọng số phạt (Cost-sensitive learning)** thông qua tham số `class_weight='balanced'`. Cách này sẽ không thay đổi dữ liệu gốc, mà trực tiếp điều chỉnh thuật toán: phạt rất nặng khi mô hình đoán sai các lớp hiếm (0, 3) và châm chước hơn với lớp đa số (1), từ đó kéo lại sự công bằng trong hàm mất mát.

### **4.2. Xây dựng mô hình**

#### **Random Forest Classifier**

In [ ]:
# Khởi tạo mô hình
rf_clf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbose=0,
)

# Train mô hình
rf_clf.fit(X_train, y_train_cls)

In [ ]:
# Dự đoán
y_pred_rf = rf_clf.predict(X_test)

# In báo cáo kết quả chi tiết
print("\n---------- KẾT QUẢ RANDOM FOREST ----------")
# Phân tích theo từng lớp 0, 1, 2, 3
print(classification_report(y_test_cls, y_pred_rf, labels=[0, 1, 2, 3]))

**Đánh giá mô hình Random Forest (Baseline) và Đặc thù dữ liệu chuỗi thời gian:**

- Dựa trên kết quả Classification Report ban đầu (Accuracy: 76%), ta rút ra 3 nhận định quan trọng về bản chất dữ liệu và hành vi của mô hình:

1. **Sự thiếu vắng của Lớp 0 (Good - Support = 0):**
Test Set hoàn toàn không chứa ngày nào có chất lượng không khí Tốt. Đây không phải là lỗi khi chia dữ liệu, mà do nhóm đã tuân thủ nguyên tắc xử lý chuỗi thời gian (`shuffle=False` để tránh rò rỉ dữ liệu tương lai - Data Leakage). Đoạn 20% dữ liệu cuối cùng rơi trúng vào thời kỳ thời tiết diễn biến xấu, phản ánh chân thực sự phân bổ khí hậu theo mùa của TP.HCM.

2. **Điểm mù nguy hiểm tại Lớp 3 (Unhealthy - Recall = 0.00):**
Mặc dù đã sử dụng `class_weight='balanced'`, mô hình vẫn dự đoán sai toàn bộ 5 ngày ô nhiễm đạt mức cao nhất. Điều này cho thấy cấu hình cây mặc định (`max_depth=12`) chưa đủ linh hoạt để cô lập được các điểm dữ liệu dị biệt này. Việc không dự đoán trước được các ngày ô nhiễm nặng là rủi ro lớn nhất cần khắc phục của hệ thống.

3. **Sự đánh lừa của Accuracy:**
Độ chính xác 76% thực chất là kết quả của việc mô hình đoán bừa Lớp 1 (Moderate - chiếm đa số). Chỉ số phản ánh đúng hiệu năng tổng thể lúc này là **Macro F1 (chỉ đạt 34%)**.

> **Do đó, phương án đưa ra:** Tiến hành tinh chỉnh siêu tham số (Hyperparameter Tuning) bằng Grid Search. Thay vì tối ưu Accuracy, ta sẽ sử dụng `scoring='f1_macro'` để ép mô hình phải tìm mọi cách nhận diện bằng được Lớp 3, chấp nhận hy sinh một phần nhỏ độ chính xác của Lớp 1 nhằm mang lại giá trị cảnh báo thực tiễn cao hơn.

In [ ]:
# # 1. Định nghĩa mô hình gốc
# rf_base = RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1)

# # 2. Grid để dò
# param_grid = {
#     "n_estimators": [200, 300, 500],
#     "max_depth": [10, 15, 20, None],
#     "min_samples_split": [2, 5, 10],
#     "min_samples_leaf": [1, 2, 4],
# }

# # 3. Vì chia tới 5 folds mà dữ liệu quá mất cân bằng nên thêm stratify vào cho các class đều ở 5 fold
# skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# # 4. Grid Search
# grid_search = GridSearchCV(
#     estimator=rf_base,
#     param_grid=param_grid,
#     cv=skf,
#     scoring="f1_macro",
#     verbose=1,
#     n_jobs=-1,
# )

# # 5. Fit
# grid_search.fit(X_train, y_train_cls)

**Lưu ý:** Sẽ không chạy lại phần GridSearch cho Random Forest vì rất rất tốn thời gian (Từ 30p - 1h30). Nhóm đã chạy sẵn và chọn ra được bộ tham số ổn định, tối ưu cho mô hình này bao gồm:
- n_estimators = 300
- max_depth = 10
- min_samples_leaf = 4
- min_samples_split = 2

In [ ]:
# Khởi tạo lại mô hình với bộ tham số tối ưu
best_params = {
    "n_estimators": 300,
    "max_depth": 10,
    "class_weight": "balanced",
    "min_samples_split": 2,
    "min_samples_leaf": 4,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": 0,
}

best_rf = RandomForestClassifier(**best_params)

# Train mô hình
best_rf.fit(X_train, y_train_cls)

In [ ]:
y_pred_best_rf = best_rf.predict(X_test)

print("\n---------- KẾT QUẢ RANDOM FOREST SAU KHI GRID SEARCH ----------")
print(classification_report(y_test_cls, y_pred_best_rf, labels=[0, 1, 2, 3]))

**Kết luận sau quá trình Tune Hyperparameter:**

- **Nguyên lý đánh đổi (Trade-off):** Lưới Grid Search đã lựa chọn một kiến trúc cây phòng thủ (`max_depth=10`, `min_samples_leaf=4`) để chống Overfitting. Thuật toán từ chối việc chia nhánh quá sâu để "học vẹt" 5 mẫu Lớp 3 dị biệt.

- **Giới hạn của Random Forest:** Dù đã áp dụng `class_weight='balanced'`, thuật toán Bagging vẫn cho thấy sự đuối sức trong việc đẩy ranh giới quyết định vươn tới các cực trị hiếm gặp của dữ liệu chuỗi thời gian phân bố bất đối xứng.


#### **XGBoost Classifier**

**Chiến lược xử lý Imbalanced Data với XGBoost:**

Khác với Random Forest, thuật toán XGBoost không hỗ trợ trực tiếp tham số `class_weight` cho bài toán phân loại đa lớp (Multi-class classification). Thay vào đó, nhóm sử dụng hàm `compute_sample_weight` của scikit-learn để tính toán trọng số trừng phạt cho từng điểm dữ liệu riêng lẻ trong tập Train.

Những ngày thuộc Lớp 3 (Unhealthy - cực hiếm) sẽ được gán một trọng số cực lớn. Khi mô hình XGBoost xây dựng các cây nối tiếp nhau (Boosting), nó sẽ bị ép phải "nhìn thấy" và tập trung sửa sai cho những mẫu có trọng số khổng lồ này.

In [ ]:
# 1. Tính trọng số cho từng dòng
# Lớp nào càng ít, dòng dữ liệu đó sẽ có trọng số càng cao
weights = compute_sample_weight(class_weight="balanced", y=y_train_cls)

# 2. Khởi tạo mô hình
xgb_clf = XGBClassifier(
    objective="multi:softmax",
    num_class=4,
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1,
)

# 3. Fit model (Ép mô hình học theo trọng số)
xgb_clf.fit(X_train, y_train_cls, sample_weight=weights)

In [ ]:
# Dự đoán
y_pred_xgb = xgb_clf.predict(X_test)

# In báo cáo kết quả chi tiết
print("\n---------- KẾT QUẢ XGBOOST ----------")
print(classification_report(y_test_cls, y_pred_rf, labels=[0, 1, 2, 3]))

**Đánh giá mô hình XGBoost và Ràng buộc Thuật toán (Algorithmic Constraints):**

- Dựa trên báo cáo phân lớp, mô hình XGBoost Classifier đã bộc lộ những giới hạn rõ rệt khi đối mặt với dữ liệu mất cân bằng cực đoan (lớp 3 và lớp 0 chỉ chiếm rất rất nhỏ):

1. **Hiệu ứng triệt tiêu lớp thiểu số (Recall Lớp 3 = 0.00):** Mặc dù nhóm đã thiết lập mảng trọng số `sample_weight` nhằm ép thuật toán chú ý đến Lớp 3, XGBoost vẫn bỏ qua toàn bộ 5 ngày ô nhiễm này. Nguyên nhân cốt lõi nằm ở cơ chế cắt tỉa cành (Pruning) và kiểm soát nhiễu cực kỳ khắt khe của XGBoost. Thuật toán đã đánh giá nhóm 1% dữ liệu mang trọng số bất thường này là outliers và gạt bỏ để bảo vệ tính tổng quát của toàn bộ cây.

2. **Sự sụt giảm hiệu năng so với Random Forest:** Việc cố gắng học nối tiếp để tối ưu hóa hàm mất mát đã gây ra tác dụng phụ lên Lớp 2. Chỉ số Recall của Lớp 2 sụt giảm nghiêm trọng từ 67% (ở Random Forest) xuống chỉ còn 40%.

- **Giải thích về việc không tinh chỉnh siêu tham số (No Hyperparameter Tuning):**
Nhóm quyết định không thực hiện Grid Search cho XGBoost. Lý do là để ép XGBoost chú ý đến Lớp 3, nên buộc phải vô hiệu hóa các cơ chế phòng thủ gốc của nó (như hạ `min_child_weight` hay `gamma` về 0). Thao tác này sẽ đẩy mô hình vào trạng thái Overfitting mất kiểm soát và phá vỡ ranh giới của các lớp đa số, sự đánh đổi này là hoàn toàn không xứng đáng.

- **Hướng giải quyết:**
Chuyển trọng tâm sang thuật toán **LightGBM Classifier**. Với cơ chế mọc cây theo chiều lá (Leaf-wise growth) có khả năng cô lập các nhóm dữ liệu nhỏ rất nhạy bén, kết hợp với việc hỗ trợ tham số `class_weight='balanced'` trực tiếp từ trong lõi thuật toán đa lớp (điều mà XGBoost thiếu sót), LightGBM được kỳ vọng là điểm rơi tối ưu nhất cho bộ dữ liệu này.

#### **LightGBM Classifier**

In [ ]:
# Khởi tạo mô hình
lgbm_clf = lgb.LGBMClassifier(
    class_weight="balanced",
    n_estimators=300,
    max_depth=10,
    learning_rate=0.05,  # Học chậm lại để không bỏ sót Lớp 3
    random_state=42,
    n_jobs=-1,
    verbose=-1,  # Không hiển thị các log
)

# Train mô hình
lgbm_clf.fit(X_train, y_train_cls)

In [ ]:
# Dự đoán
y_pred_lgbm = lgbm_clf.predict(X_test)

print("\n---------- KẾT QUẢ LIGHTGBM ----------")
print(classification_report(y_test_cls, y_pred_lgbm))

**Đánh giá mô hình LightGBM Classifier:**

- Dựa trên log huấn luyện và báo cáo phân lớp, cấu hình mặc định của LightGBM đã bộc lộ những đặc tính thú vị khi xử lý dữ liệu bất đối xứng cực đoan:

1. **Khả năng nhận diện trọng số (Log Prior):** Dòng log `Start training from score -1.386294` minh chứng cho việc LightGBM đã tiếp nhận thành công lệnh `class_weight='balanced'`. Thuật toán đã chủ động ép xác suất xuất phát của cả 4 phân lớp về mức cân bằng (xấp xỉ 25% mỗi lớp, tương đương $\ln(0.25) \approx -1.386$) trước khi bắt đầu mọc cây.

2. **Cơ chế phòng thủ tự động:** Các cảnh báo `No further splits with positive gain` (đã được giải quyết bằng `verbose=-1`) không phải là lỗi, mà là cơ chế chống Overfitting xuất sắc của mô hình. Khi gặp các nhánh chứa quá ít dữ liệu (như Lớp 3), thay vì cố gắng đâm sâu để "học vẹt", thuật toán đã chủ động dừng việc phân nhánh vì nhận thấy việc này không mang lại thêm giá trị thông tin cho tổng thể.

> **Kết luận:** Thuật toán mọc cây theo lá (Leaf-wise) của LightGBM có tiềm năng rất lớn, nhưng cấu hình mặc định đang ngăn cản khả năng chạm tới các điểm dữ liệu dị biệt. Nhóm sẽ sử dụng Grid Search để mở khóa các tham số kiểm soát cấu trúc lá (như `num_leaves` và `min_child_samples`), cho phép thuật toán tạo ra các "chiếc lá nhỏ" để hứng trọn vẹn Lớp 3 mà không làm hỏng tính tổng quát.

In [ ]:
# # Khởi tạo mô hình
# lgbm_base = lgb.LGBMClassifier(
#     class_weight="balanced", random_state=42, n_jobs=-1, verbose=-1
# )

# # Không gian tham số ccho mọc cây theo lá (Leaf-wise)
# param_grid_lgbm = {
#     "n_estimators": [200, 300],
#     "learning_rate": [0.01, 0.05],
#     "max_depth": [8, 10, -1],  # -1 nghĩa là không giới hạn độ sâu
#     "num_leaves": [15, 31, 50],  # Giới hạn số lá để tránh Overfitting
#     "min_child_samples": [2, 5, 10],  # Cho phép tạo lá nhỏ để "hứng" Lớp 3
# }

# # Cấu hình Grid Search
# skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# grid_search_lgbm = GridSearchCV(
#     estimator=lgbm_base,
#     param_grid=param_grid_lgbm,
#     cv=skf,
#     scoring="f1_macro",
#     verbose=1,
#     n_jobs=-1,
# )

# # Fit
# grid_search_lgbm.fit(X_train, y_train_cls)

**Lưu ý:** Sẽ không chạy lại phần GridSearch cho LightGBM vì rất rất tốn thời gian (Từ 30p - 1h30). Nhóm đã chạy sẵn và chọn ra được bộ tham số ổn định, tối ưu cho mô hình này bao gồm:
- learning_rate = 0.01
- max_depth = -1 (Không giới hạn)
- min_child_samples = 10
- n_estimators = 300
- num_leaves = 15

In [ ]:
# Khởi tạo lại mô hình với bộ tham số tối ưu
best_params = {
    "n_estimators": 300,
    "max_depth": -1,
    "min_child_samples": 10,
    "num_leaves": 15,
    "learning_rate": 0.01,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
    "class_weight": "balanced",
}

best_lgbm = lgb.LGBMClassifier(**best_params)

# Train mô hình
best_lgbm.fit(X_train, y_train_cls)

In [ ]:
y_pred_best_lgbm = best_lgbm.predict(X_test)

print("\n---------- KẾT QUẢ LIGHTGBM SAU KHI GRID SEARCH ----------")
print(classification_report(y_test_cls, y_pred_best_lgbm, labels=[0, 1, 2, 3]))

**Đánh giá mô hình LightGBM sau tinh chỉnh và Xác định mô hình tối ưu:**

- **Phá vỡ giới hạn dữ liệu dị biệt (Anomaly Detection):**
Việc mở giới hạn cấu trúc lá (`max_depth=-1`, `min_child_samples=10`) kết hợp với tốc độ học chậm (`learning_rate=0.01`) đã tạo ra một bước đột phá. LightGBM đã vượt qua giới hạn Concept Drift, thành công kéo điểm Recall của Lớp 3 (Unhealthy) từ 0.00 lên 0.60. Điều này chứng minh thuật toán đã thực sự bóc tách được các ranh giới đặc trưng cực nhỏ của những ngày ô nhiễm nghiêm trọng.

- **Sự ổn định và Khả năng tổng quát hóa của mô hình:**
Việc Accuracy tổng thể không thay đổi là một điểm sáng cho việc Grid Search. Chỉ số đại diện cho sự cân bằng tổng thể là Macro F1 đã tăng từ 0.42 lên 0.5, xác nhận mô hình không còn bị thiên vị (bias) mà đã học được bức tranh toàn cảnh khách quan hơn.

> **Kết luận:** Với khả năng nhận diện được cả rủi ro cao nhất (Lớp 3), `LightGBM Classifier (Tuned)` được lựa chọn làm mô hình tốt nhất của dự án để triển khai các bước tiếp theo.

### **4.3. Đánh giá và so sánh**

#### **So sánh 3 mô hình**

In [ ]:
# Gom các dự đoán vào một từ điển
model_preds = {
    "Random Forest (Tuned)": y_pred_best_rf,
    "XGBoost (Based)": y_pred_xgb,
    "LightGBM (Tuned)": y_pred_best_lgbm,
}
labels = [0, 1, 2, 3]

# Tính các chỉ số
comparison_data = []
for model_name, y_pred in model_preds.items():
    acc = accuracy_score(y_test_cls, y_pred)
    precision_macro = precision_score(
        y_test_cls, y_pred, labels=labels, average="macro", zero_division=0
    )
    recall_macro = recall_score(
        y_test_cls, y_pred, labels=labels, average="macro", zero_division=0
    )
    f1_macro = f1_score(
        y_test_cls, y_pred, labels=labels, average="macro", zero_division=0
    )
    f1_weight = f1_score(
        y_test_cls, y_pred, labels=labels, average="weighted", zero_division=0
    )

    comparison_data.append(
        {
            "Mô hình": model_name,
            "Accuracy": round(acc, 4),
            "Precision (Macro)": round(precision_macro, 4),
            "Recall (Macro)": round(recall_macro, 4),
            "F1-Score (Macro)": round(f1_macro, 4),
            "F1-Score (Weighted)": round(f1_weight, 4),
        }
    )

# Hiển thị
df_compare = pd.DataFrame(comparison_data).set_index("Mô hình")
print("---------- BẢNG SO SÁNH HIỆU NĂNG 3 MÔ HÌNH CLASSIFICATION ----------")
display(df_compare)

#### **Trực quan so sánh 3 mô hình**

In [ ]:
df_plot = df_compare.T

ax = df_plot.plot(
    kind="bar", figsize=(14, 7), colormap="Set2", edgecolor="black", width=0.7
)

plt.title("So Sánh Các Mô Hình Theo Metrics", fontweight="bold", fontsize=14, pad=15)
plt.xlabel("Metrics", fontweight="bold", fontsize=11)
plt.ylabel("Score", fontweight="bold", fontsize=11)

plt.xticks(rotation=0, fontweight="bold", fontsize=11)

plt.legend(title="Mô Hình", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=11)

# In giá trị lên đầu mỗi bar
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(
            f"{height:.2f}",
            (p.get_x() + p.get_width() / 2.0, height),
            ha="center",
            va="bottom",
            fontsize=10,
            color="black",
            fontweight="bold",
            xytext=(0, 4),
            textcoords="offset points",
        )

plt.ylim(0, 1.15)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.grid(False)
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "all_model_classifier_compare")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

**Đánh giá và So sánh tổng thể 3 thuật toán Phân lớp:**

- Dựa trên Bảng tổng hợp toàn diện các chỉ số, nhóm rút ra quyết định lựa chọn mô hình dựa trên đặc thù bài toán môi trường:

**1. Sự mơ hồ của Accuracy và Weighted Score:**

Nếu chỉ nhìn vào `Accuracy` (~77-78%) hay `F1-Weighted` (~77-78%), cả XGBoost và LightGBM đều trông có vẻ xuất sắc tương đương nhau. Tuy nhiên, các chỉ số này đã bị thao túng bởi lớp đa số (Lớp 1 - chiếm 76.5% dữ liệu). Một hệ thống cảnh báo rủi ro không thể chỉ giỏi đoán những ngày bình thường mà bỏ qua những ngày rất ô nhiễm (nguy hiểm cho sức khỏe).

**2. Sự lựa chọn dựa trên chỉ số Macro (Sự công bằng):**
Khi đánh giá bằng Macro (quyền lợi của cả 4 lớp bằng nhau), bức tranh thực sự mới lộ diện:

  - **XGBoost & Random Forest:** Đều mắc kẹt ở mức `Recall (Macro)` thấp (~37-38%) do hoàn toàn không dự đoán được 5 ngày ô nhiễm cực đoan (Lớp 3).

  - **LightGBM (Tuned):** Làm tốt nhất với `Recall (Macro)` đạt **52%** và `F1-Macro` đạt **50%**. Nhờ việc mở giới hạn cấu trúc lá (`max_depth=-1`, `min_child_samples=10`), LightGBM là thuật toán duy nhất chẻ sâu để đoán được Lớp 3 (Unhealthy).

**3. Sự đánh đổi (Trade-off):**

Việc LightGBM hy sinh một chút `Precision` ở Lớp 1 để đổi lấy khả năng cảnh báo `Recall` ở Lớp 3 là một sự đánh đổi hoàn toàn có chủ đích và mang giá trị thực tiễn cao. Thà báo động nhầm để người dân phòng bị, còn hơn bỏ sót những ngày không khí có độ ô nhiễm cao.

> **Kết luận:** Với khả năng bóc tách dữ liệu dị biệt tốt nhất, **LightGBM Classifier (Tuned)** được chọn làm Best Classifier để thực hiện các bước tiếp.

### **4.4. Confusion Matrix**

In [ ]:
display_labels = ["Good (0)", "Moderate (1)", "Sensitive (2)", "Unhealthy (3)"]

fig, ax = plt.subplots()

ConfusionMatrixDisplay.from_predictions(
    y_test_cls,
    y_pred_best_lgbm,
    display_labels=display_labels,
    cmap="Oranges",
    colorbar=False,
    ax=ax,
)

ax.set_title(
    "Confusion Matrix - Best Classifier (Tuned LightGBM)",
    fontweight="bold",
    fontsize=14,
    pad=15,
)
ax.set_xlabel("Predicted", fontweight="bold")
ax.set_ylabel("Actual", fontweight="bold")
ax.grid(False)

# Vẽ đường kẻ chia ô
n = len(display_labels)
ax.set_xticks([x - 0.5 for x in range(1, n)], minor=True)
ax.set_yticks([y - 0.5 for y in range(1, n)], minor=True)
ax.grid(which="minor", color="black", linewidth=0.5)

plt.tight_layout()
models_dir = os.path.join(figures_dir, "models")
file_path = os.path.join(models_dir, "confusion_matrix")
plt.savefig(file_path, bbox_inches="tight")
print("Đã lưu biểu đồ thành công")
plt.show()

### **4.5. Hành động thực tiễn**

Mô hình phân loại chất lượng không khí không chỉ dừng lại ở các con số chính xác trên tập Test, mà cấu trúc đầu ra của mô hình (Lớp 0, 1, 2, 3) được ánh xạ trực tiếp thành các khuyến nghị hành động cụ thể trong thực tế đời sống và quản lý đô thị tại TP.HCM:

**Mức Tốt (Good - Lớp 0):**

- Ý nghĩa: Chất lượng không khí lý tưởng, không gây rủi ro cho sức khỏe.

- Hành động:
    - Khuyến khích người dân tích cực tham gia các hoạt động thể thao, giải trí ngoài trời.
    - Các trường học ưu tiên tổ chức các buổi dã ngoại, thể dục ngoại khóa cho học sinh.
    - Hệ thống thông gió của các tòa nhà cao tầng tăng cường lấy khí tươi tự nhiên từ bên ngoài để tiết kiệm điện năng.

**Mức Trung bình (Moderate - Lớp 1):**

- Ý nghĩa: Chất lượng không khí ở mức chấp nhận được nhưng bắt đầu xuất hiện rủi ro nhỏ đối với nhóm người nhạy cảm.

- Hành động:
    - Người dân nói chung vẫn có thể hoạt động ngoài trời bình thường.
    - Nhóm nhạy cảm (trẻ em, người già, người có bệnh nền hen suyễn, tim mạch) nên giảm các hoạt động gắng sức hoặc kéo dài ngoài trời; cân nhắc đóng bớt cửa sổ nếu vị trí nhà ở gần các trục đường giao thông lớn.

**Mức Kém / Nguy hiểm (Unhealthy - Lớp 2 & 3):**

- Ý nghĩa: Không khí chuyển sang ngưỡng có hại, gây ảnh hưởng trực tiếp đến hệ hô hấp của toàn bộ cộng đồng.

- Hành động ứng phó:
    - Đối với người dân: Yêu cầu bắt buộc đeo khẩu trang đạt chuẩn chống bụi mịn (N95, PM2.5) khi di chuyển ngoài đường. Các hộ gia đình đóng kín cửa sổ, bật máy lọc không khí ở chế độ công suất cao. Hạn chế tối đa các hoạt động thể dục buổi sáng ngoài trời.
    - Đối với quản lý đô thị: Kích hoạt ngay hệ thống xe phun nước dập bụi tại các đại lộ lớn; điều tiết phân luồng giao thông từ xa để giảm thiểu hiện tượng kẹt xe giờ cao điểm; tạm dừng các hoạt động thi công đào đường hoặc xây dựng lộ thiên phát tán nhiều bụi cho đến khi chỉ số AQI hạ nhiệt.

### **4.6. Tổng kết**

#### **Lý do lựa chọn LightGBM Classifier (Tuned) làm mô hình tối ưu nhất**

- **Nhận diện phân lớp ô nhiễm cực đoan (Lớp 3):** Mặc dù Random Forest (Tuned) có điểm số F1-Macro tổng thể cao hơn, nhưng thuật toán này bỏ sót hoàn toàn các ngày thuộc Lớp 3 (Mức Nguy hại). Trong bài toán sức khỏe cộng đồng, việc bỏ sót ngày ô nhiễm cực đoan mang lại rủi ro lớn hơn việc dự báo nhầm. LightGBM với cơ chế phát triển cây theo cấu trúc lá (Leaf-wise) là mô hình duy nhất nhận diện được Lớp 3.

- **Sự ổn định trên các phân lớp khác:** Mô hình duy trì độ chính xác và độ bao phủ cao ở Lớp 1, đồng thời giữ được mức cân bằng đạt yêu cầu ở Lớp 2.

#### **Đánh giá hạn chế của dự án**

- **Sự mất cân bằng dữ liệu:** Số lượng mẫu Lớp 3 trong tập kiểm thử quá ít (5 dòng dữ liệu) khiến chỉ số F1-Macro chung của LightGBM bị kéo thấp xuống do mô hình chưa có đủ thông tin để học các đặc trưng cực đoan.

- **Thiếu hụt đặc trưng khí tượng:** Ranh giới phân loại bị nhiễu do tập dữ liệu chỉ chứa nồng độ chất ô nhiễm quá khứ. Việc thiếu các biến động lực học như tốc độ gió, hướng gió, độ ẩm hoặc lượng mưa khiến cấu trúc cây quyết định gặp khó khăn khi thời tiết biến động đột ngột.

#### **Hướng phát triển tiếp theo**

- **Xử lý dữ liệu lệch:** Áp dụng kỹ thuật tái lấy mẫu SMOTE để tự động sinh thêm mẫu giả lập cho Lớp 3, tăng không gian huấn luyện cho LightGBM.

- **Làm giàu đặc trưng:** Tích hợp các yếu tố thời tiết thực tế thực hiện qua OpenWeatherMap API (gió, mưa, độ ẩm) nhằm làm rõ ranh giới phân loại giữa các cấp độ AQI.

- **Triển khai ứng dụng:** Đóng gói mô hình LightGBM đã tối ưu hóa thành công cụ dự báo và xây dựng giao diện bảng điều khiển trực quan bằng Streamlit để hỗ trợ theo dõi thực tế.

#### **Lưu best classifier**

In [ ]:
# Lưu best model chính thức
best_model = best_lgbm

models_dir = os.path.join(ROOT, "models")
os.makedirs("models", exist_ok=True)
save_path = os.path.join(models_dir, "best_classifier.pkl")
joblib.dump(best_model, save_path)

print("Đã lưu best_classifier.pkl")